In [0]:
from pyspark.sql import functions as F

lake = "abfss://bronze@stagentops.dfs.core.windows.net"

# --- Traces: incremental load with Auto Loader (idempotent via checkpoint) ---
stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{lake}/_schemas/agent_traces")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(f"{lake}/raw/traces/")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp()))

(stream.writeStream
    .option("checkpointLocation", f"{lake}/_checkpoints/agent_traces")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("dbw_agentops.bronze.agent_traces")
    .awaitTermination())

# --- Pricing: small reference file, batch overwrite (idempotent) ---
pricing = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{lake}/raw/pricing/")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp()))
pricing.write.mode("overwrite").saveAsTable("dbw_agentops.bronze.model_pricing")